# Assignment: My Data is a Mess. Can I Trust Any of It?

**Course:** Quantitative Data Analysis (INFOMQUDA)  
**Environment:** Google Colab (CPU is sufficient)  
**Datasets:** Synthetic heart-rate data (Sections 1–2) · Gapminder (Sections 3–6)

---

## Requirements for this practical

For the first tutorial session, you are expected to complete **Sections 1–2**.

The remaining sections will be made available before the next tutorial.

In these practicals, you should be able to:

1. understand what each code cell is doing;
2. change parameters and explain how the results change;
3. reproduce the main parts of the code yourself;
4. answer the questions at the end of each section;
5. ask questions during the practical if anything is unclear.

Questions asked during the practical will be noted together with attendance.

In every section, the questions are placed **at the end of the section**.


## Overview

Before you can trust any analysis, you need to trust your data. This assignment works through the most common ways data can mislead you — and what to do about it.

You will:

1. **See** how noise can hide a real signal — and how denoising recovers it  
2. **Understand** outliers: when to remove them, and what they reveal about your sample  
3. **Explore** data distributions and what "normal" really means  
4. **Transform** data and learn when that helps — and when it creates new problems
5. **Question** when to trust a significant relationship and when to dig deeper
6. **Test** hypotheses correctly, including what to do when you test many at once  
7. **Sample** wisely and understand what your results can and cannot generalise to  

---

## How to use this notebook

- **Explanatory cells** introduce each concept and explain *why* it matters.  
- **Code cells** contain working implementations — read them before running.  
- Cells marked **STUDENT TASK** ask you to adjust parameters or input your own code.  
- Questions marked **Q** should be answered in the notebook (edit the `> Your answer here` box).  

> **Colab note:** Run cells top-to-bottom. If your runtime disconnects, restart from relevant upper Sections.

---

## Table of Contents

| Section | Topic |
|---------|-------|
| [0. Setup](#setup) | Packages, imports, reproducibility |
| [1. Noise](#noise) | When noise hides a real relationship |
| [2. Outliers](#outliers) | When outliers change real results |
| [3. Distributions](#distributions) | What does the data actually look like? |
| [4. Transformations](#transformations) | Useful — but dangerous |
| [5. Misleading relationships](#misleading) | Can we still be missing something? |
| [6. Hypothesis testing](#testing) | Significance tests and multiple comparisons |
| [7. Sampling](#sampling) | Who is in your data, and who is not? |


---
## Section 0: Setup <a id='setup'></a>

We install and import everything once here. All subsequent sections assume these have run.

**Packages used:**

| Package | Purpose |
|---------|---------|
| `numpy`, `pandas` | Arrays and data frames |
| `matplotlib`, `seaborn` | Plotting |
| `scipy.stats` | Statistical tests |
| `sklearn` | Preprocessing, models, splits, metrics |
| `torch` | Neural network (refresher in this section) |


In [ ]:
# ── 0.1  Install packages ────────────────────────────────────────────────────
# Run this first. Output is suppressed for readability.
!pip install -q numpy pandas matplotlib seaborn scipy scikit-learn torch statsmodels > /dev/null 2>&1
print("Packages ready.")


In [ ]:
# ── 0.2  Imports and global settings ─────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import zscore
from statsmodels.stats.multitest import multipletests
from itertools import combinations
import warnings
warnings.filterwarnings("ignore")

# Reproducibility
SEED = 36
np.random.seed(SEED)

# Plot style — clean, minimal
plt.rcParams.update({
    "figure.dpi":        110,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "font.size":         11,
})

print("Imports done.")


### Scikit-learn refresher

scikit-learn is the standard Python library for machine learning and data preprocessing.  

Three ideas are worth recalling before we start, because we will use all three later.

**The fit / transform pattern**
```python
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaler.fit(X_train)               # learns mean and std from training data only
X_scaled = scaler.transform(X_train)   # applies the learned transformation
```
Always `fit` on training data and `transform` everything else with the *same fitted object*. Fitting on the full dataset before splitting is one of the most common data-leakage mistakes.

**train_test_split**
```python
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
```
`random_state` makes the split reproducible. For time-series or repeated-measures data, random splitting is *wrong* — we return to this in Section 7.

**A simple pipeline**
```python
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model",  LogisticRegression()),
])
pipe.fit(X_train, y_train)
pipe.predict(X_test)
```
Pipelines chain preprocessing and modelling so fitting and transforming always happen in the correct order, even inside cross-validation.


In [ ]:
# CODE HERE START
# Task:
# 1. Create x values from 0 to 19.
# 2. Create y values using approximately: y = 2*x + noise.
# 3. Fit a LinearRegression model.
# 4. Print the slope.

# CODE HERE END

### PyTorch refresher

PyTorch is the standard deep-learning library. We use neural networks in module 3. Three concepts are worth remembering.

**Tensors** — PyTorch's equivalent of NumPy arrays, supporting GPU and automatic differentiation:
```python
import torch
x = torch.tensor([[1.0, 2.0], [3.0, 4.0]])    # shape (2, 2)
x = torch.from_numpy(numpy_array).float()      # convert from NumPy
```

**A minimal neural network**
```python
import torch.nn as nn

class SimpleNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)   # output shape: (batch,)
```

**A minimal training loop** — these four steps are always in this order:
```python
model     = SimpleNet(input_dim=10)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

for epoch in range(100):
    model.train()
    pred  = model(X_tensor)         # 1. forward pass
    loss  = criterion(pred, y_tensor)   # 2. compute loss
    optimizer.zero_grad()           # 3. clear old gradients
    loss.backward()                 # 4. backpropagate
    optimizer.step()                # 5. update weights
```


**Device** — For this practical, we use the **CPU**. CUDA/GPU is not needed for this notebook. To use CUDA, you would need to select a GPU runtime in the notebook settings and change the device line in the code. This may use more computing credits and is unnecessary for this practical.

In [ ]:
# ── 0.3  Quick PyTorch test ────────────────────────────────────────────
import torch
import torch.nn as nn

DEVICE = device = torch.device("cpu")
print(f"PyTorch {torch.__version__} · device: {DEVICE}")

class SimpleNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

# CODE HERE START
# 1. define simplenet
# 2. create a small dataset
# 3. apply simple net to this data
# 4. print the shape of the data and output

# CODE HERE END


---
# PART I. Synthetic Heart Rate Dataset - Does exam stress lead to high heart rate?

We generate synthetic heart-rate (HR) data simulating students just before an exam. We assume that **low stress** before an exam leads to lower HR and **high stress** to higher HR.

Heart rate is often measured in **BPM**, which means **beats per minute**.

Important: this is **not an ECG simulation**.

An ECG signal shows electrical activity of the heart and contains sharp R-peaks. Here, we are already working with a simplified heart-rate time series: each value is an estimated HR in BPM.

So this notebook simulates something closer to:

> “The wearable estimated that the person’s HR was 72 BPM at this time point.”

not:

> “This is the raw electrical waveform of the heart.”

The simulation is intentionally simple and limited. Its purpose is teaching, not physiological realism.

---
## Section 1: When Noise Hides a Real Relationship <a id='noise'></a>

High stress raises HR by a fixed amount — there *is* a real difference in the data. But when we add enough measurement noise, that difference can become invisible to a statistical test.

This section shows:
- How signal-to-noise ratio affects whether you can detect a true effect  
- Two denoising strategies and how well each recovers the original signal  
- Why denoising is powerful but can also create artefacts if misapplied  

**Synthetic data** — We simulate HR using a sine wave:

$$\text{HR drift}(t) = A \sin(2\pi f t)$$

where:

- \(A\) is the amplitude of the drift;
- \(f\) is the frequency;
- \(t\) is time.

We also simulate several types of variation in the data:
1. Between-participant variation: we draw from a normal distribution with mean of true HR and standad deviation of PARTICIPANT_VARIABILITY
2. Within participant variation: HR_DRIFT_AMPLITUDE, HR_DRIFT_FREQUENCY
3. Small random time-point noise: independent fluctuation at every measurement, drawn from a normal distribution with standard deviation of TIME_POINT_NOISE BPM
4. Artefact bursts: simulating movement, slipped sensor, or bad contact. Each participants gets between N_ARTIFACT_MIN and N_ARTIFACT_MAX episodes of amplitude ARTIFACT_BURST, each lasting ARTIFACT_MIN_LENGTH to ARTIFACT_MAX_LENGTH seconds.


In [ ]:
# ── 1.1  Synthetic HR data generator ─────────────────────────────────────────

# STUDENT TASK: adjust the parameters below and re-run the cells

N_SAMPLES   = 20      # participants per condition
HR_LOW      = 70      # group mean HR, low stress
HR_HIGH     = 75      # group mean HR, high stress

DURATION    = 5.0     # in minutes
FS          = 60      # number of measurements per minute
PARTICIPANT = 5       # participant to inspect

PARTICIPANT_VARIABILITY = 5
HR_DRIFT_AMPLITUDE = 3
HR_DRIFT_FREQUENCY = 0.1

N_ARTIFACT_MIN = 1
N_ARTIFACT_MAX = 4
ARTIFACT_MIN_LENGTH = 20  # seconds
ARTIFACT_MAX_LENGTH = 80  # seconds
TIME_POINT_NOISE = 5
ARTIFACT_BURST = 25      # in BPM

# ─────────────────────────────────────────────────────────────────────────────

t = np.linspace(0, DURATION, int(DURATION * FS))
T = len(t)

def make_hr_signals(n_samples, seed=369):
    np.random.seed(seed)
    signals = {}
    for condition, true_hr in [("low", HR_LOW), ("high", HR_HIGH)]:

        # CODE HERE START
        # 1. Between-participant variation: give each of the n_samples participants
        #    their own baseline HR, drawn from a normal distribution centred on
        #    true_hr with a standard deviation of PARTICIPANT_VARIABILITY.
        # 2. Within-participant drift: build a slow sinusoid over t with amplitude
        #    HR_DRIFT_AMPLITUDE and frequency HR_DRIFT_FREQUENCY based on the function
        #    given above.

        # between_p =
        # within_p =

        # CODE HERE END

        clean     = between_p[:, None] + within_p[None, :]
        noisy     = clean.copy()

        # 3. Small random noise at every single measurement.
        wobble = np.random.randn(n_samples, T) * TIME_POINT_NOISE

        # 4. Artefact bursts: sensor reads too high.
        artefacts = np.zeros((n_samples, T))
        for i in range(n_samples):
            for _ in range(np.random.randint(N_ARTIFACT_MIN, N_ARTIFACT_MAX)):
                length = np.random.randint(ARTIFACT_MIN_LENGTH, ARTIFACT_MAX_LENGTH)
                start  = np.random.randint(0, T - length)
                mag    = np.abs(np.random.randn()) * ARTIFACT_BURST
                artefacts[i, start:start + length] += mag


        noisy = clean + wobble + artefacts

        signals[condition] = (clean, noisy)
    return signals

signals = make_hr_signals(N_SAMPLES)
clean_low,  noisy_low  = signals["low"]
clean_high, noisy_high = signals["high"]

print(f"Generated {N_SAMPLES} participants per condition")
print(f"Noise level:        {ARTIFACT_BURST} BPM")
print(f"True HR difference: {HR_HIGH - HR_LOW} BPM")
print(f"Signal shape:       {clean_low.shape}  (participants × time-points)")

In [ ]:
# ── 1.2  Single-participant view: clean vs noisy ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), sharey=True)

for ax, data, label in zip(
        axes,
        [clean_low, noisy_low],
        ["Clean signal — participant 5, low stress",
         "Noisy signal — participant 5, low stress"]):
    ax.plot(t, data[PARTICIPANT-1], lw=1.2, color="steelblue")
    ax.set_title(label, fontsize=10)
    ax.set_xlabel("Time (minutes)")
    ax.set_ylabel("Heart rate (BPM)")

plt.tight_layout()
plt.show()


In [ ]:
# ── 1.3  Group means: Can you see the difference with noise? ─────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for ax, (cl, ch), label in zip(
        axes,
        [(clean_low, clean_high), (noisy_low, noisy_high)],
        ["Clean data", "Noisy data"]):
    for sig, col in zip([cl, ch], ["steelblue", "tomato"]):
        for row in sig:
            ax.plot(t, row, alpha=0.15, lw=0.6, color=col)
    ax.plot(t, cl.mean(0), color="steelblue", lw=2, label="Low stress mean")
    ax.plot(t, ch.mean(0), color="tomato",    lw=2, label="High stress mean")
    ax.set_title(label, fontsize=11)
    ax.set_xlabel("Time (minutes)")
    ax.set_ylabel("Heart rate (BPM)")
    ax.legend()

plt.tight_layout()
plt.show()


In [ ]:
# ── 1.4  Statistical test: does the group difference survive noise? ────────────
# We have N participants per condition, each with a full time series.
# The t-test assumes independent observations.
#
# Hence, we summarise each participant's recording into ONE number,
# then compare those numbers across groups with a t-test.
# The summary you choose can determine what effect you can detect.

def summarise(data):
    """data shape: (n_participants, n_timepoints)"""
    mean_hr      = data.mean(axis=1)                  # average level
    peak_hr      = data.max(axis=1)                   # highest point reached
    return mean_hr, peak_hr

low_clean_mean,  low_clean_peak  = summarise(clean_low)
high_clean_mean, high_clean_peak = summarise(clean_high)
low_noisy_mean,  low_noisy_peak  = summarise(noisy_low)
high_noisy_mean, high_noisy_peak = summarise(noisy_high)

print(f"{'Summary':<18}  {'clean p':>9}  {'noisy p':>9}")
print("─" * 40)
for label, cl, ch, nl, nh in [
    ("Mean HR",       low_clean_mean, high_clean_mean, low_noisy_mean, high_noisy_mean),
    ("Peak HR",       low_clean_peak, high_clean_peak, low_noisy_peak, high_noisy_peak)
]:
    _, p_clean = stats.ttest_ind(cl, ch)
    _, p_noisy = stats.ttest_ind(nl, nh)
    print(f"{label:<18}  {p_clean:>9.4f}  {p_noisy:>9.4f}")


### 1.5 Denoising strategies

Denoising means trying to recover the underlying signal from noisy measurements.

Different methods solve different problems, for example:

| Method | What it does | Works best for |
|---|---|---|
| Rolling mean | Averages neighbouring values | small random noise |
| Convolution smoother | Same idea as rolling mean, written as a filter | small random noise |
| Sensor clean-up | Removes implausible readings and interpolates across them | sensor artefacts |

For more background see: https://www.dspguide.com/ch15.htm

In [ ]:
# ── 1.5  Two denoising methods ─────────────────────────────────────────────
# STUDENT TASK: try different window sizes and thresholds
ROLLING_WINDOW = 15    # seconds
MAD_THRESHOLD  = 2.5   # how many robust SDs before a point counts as artefact

# ── Method 1: Rolling mean ────────────────────────────────────────────────────
# Replaces each timepoint with the average of its neighbours.
def rolling_mean(signal, window=ROLLING_WINDOW):
    # CODE HERE START
    # 1. define rolling mean denoising: replace each timepoint with the mean of the surrounding `window` points.

    # CODE HERE END

# ── Method 2: Artefact rejection ─────────────────────────────────────────────
# Detects implausible timepoints and interpolates across them.
def artefact_rejection(signal, threshold=MAD_THRESHOLD):
    med  = np.median(signal)

    # MAD = median absolute deviation: a spread measure that ignores outliers,
    # unlike the standard deviation, which the artefacts themselves would inflate.
    mad  = np.median(np.abs(signal - med))

    # For normally distributed data, MAD * 1.4826 equals the standard deviation.
    # Multiplying by that constant lets us read the threshold as "2.5 SDs",
    # but computed in a way the artefacts cannot distort.
    good = np.abs(signal - med) < threshold * mad * 1.4826 # define good signal

    cleaned = signal.copy()
    cleaned[~good] = np.interp(np.flatnonzero(~good), np.flatnonzero(good), signal[good]) # interpolate across bad timepoints
    return cleaned

DENOISING = {
    "Rolling mean":       rolling_mean,
    "Artefact rejection": artefact_rejection,
}

def apply_smoother(data_array, fn):
    return np.stack([fn(row) for row in data_array])

denoised = {
    name: (apply_smoother(noisy_low, fn), apply_smoother(noisy_high, fn))
    for name, fn in DENOISING.items()
}

print("All denoising methods applied.")


In [ ]:
# ── 1.6  Side-by-side: original · noisy · two denoising methods ────────────
panels = (
    [("Original clean data", clean_low,  clean_high),
     ("Noisy data",          noisy_low,  noisy_high)] +
    [(name, dl, dh) for name, (dl, dh) in denoised.items()]
)

fig, axes = plt.subplots(2, 2, figsize=(15, 7), sharey=True)

for ax, (title, dl, dh) in zip(axes.flat, panels):
    for sig, col in zip([dl, dh], ["steelblue", "tomato"]):
        for row in sig:
            ax.plot(t, row, alpha=0.12, lw=0.5, color=col)
    ax.plot(t, dl.mean(0), color="steelblue", lw=2, label="Low stress mean")
    ax.plot(t, dh.mean(0), color="tomato",    lw=2, label="High stress mean")
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Time (min)")
    ax.set_ylabel("HR (BPM)")

axes[0, 0].legend(fontsize=8)
plt.suptitle("Original · Noisy · Two denoising methods", fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── 1.7  t-test: which method recovers the real effect? ──────────────────────
_, p_clean = stats.ttest_ind(clean_low.mean(1), clean_high.mean(1))

print(f"{'Method':<25}  {'clean p':>9}  {'denoised p':>11}")
print("─" * 63)

for name, (dl, dh) in denoised.items():
    _, p_denoised = stats.ttest_ind(dl.mean(1), dh.mean(1))
    print(f"{name:<25}  {p_clean:>9.4f}  {p_denoised:>11.4f}")

**Q1 —**

a. Increase/Decrease `ARTIFACT_BURST` and other noise variables. What happens to the visual separation?

b. Change `N_SAMPLES`. Does a larger sample size help even when noise level is high?

c. Which of the two denoising methods recovers the group means in your experiment? Try to explain intuitively why.

d. Describe a scenario where applying a smoother would *create* a false signal that was never present in the raw data.

> *Your answer here*

---
## Section 2: When Outliers Change Real Results <a id='outliers'></a>

Outliers come in different types, and the right decision for each can be different:

| Type | Example | Right action |
|------|---------|-------------|
| **Measurement error** | Sensor glitch records 200 BPM | Remove — not a real observation |
| **Real but rare in our data** | Elderly participant with much lower HR | Think carefully — removing it changes your population |

This section shows:
- How adding both types of outliers changes the statistical conclusions  
- How to identify outliers  
- How to remove outliers


In [ ]:
# ── 2.1  Add outliers to the groups ──────────────────────────────────────────

# STUDENT TASK: try changing N_ELDERLY and N_SENSOR and re-run.

N_ELDERLY = 3     # elderly participants in high-stress group
N_SENSOR  = 3     # sensor failures happening in low-stress group
PARTICIPANT_VARIABILITY = 5

# ─────────────────────────────────────────────────────────────────────────────

np.random.seed(SEED)

# Reuse clean data from Section 1
base_low  = clean_low.mean(axis=1)
base_high = clean_high.mean(axis=1)

# Elderly: real people, genuinely lower HR due to age.
# In our dataset, they "happen to" all have high-stress before the exam.
elderly_high  = 60 + np.random.randn(N_ELDERLY) * PARTICIPANT_VARIABILITY

# Sensor errors: physiologically impossible values from a broken device (200+ BPM).
# In our dataset, the faulty sensor "happened to be" worn only by low-stress individuals.
sensor_low    = np.random.uniform(200, 230, size=N_SENSOR)

# add the outliers to the clean data
high_with_elderly = np.concatenate([base_high, elderly_high])
low_with_sensor   = np.concatenate([base_low,  sensor_low])

DATASETS = {
    "Original":            (base_low,         base_high),
    "Elderly added":       (base_low,          high_with_elderly),
    "Sensor errors added": (low_with_sensor,   base_high),
    "Both added":          (low_with_sensor,     high_with_elderly),
}

print(f"{'Dataset':<25}  {'n (low/high)':>13}  {'t':>6}  {'p':>8}  {'Sig?'}")
print("─" * 65)
for name, (lo, hi) in DATASETS.items():
    t_val, p_val = stats.ttest_ind(lo, hi)
    sig = "Yes" if p_val < 0.05 else "No"
    print(f"{name:<25}  {len(lo):>5}/{len(hi):<5}  {t_val:>6.2f}  {p_val:>8.3f}  {sig}")


In [ ]:
# ── 2.2  Boxplots: original · with outliers · after IQR removal ─────────────
def iqr_remove(arr):
    q1, q3 = np.percentile(arr, [25, 75])
    fence   = 1.5 * (q3 - q1)
    return arr[(arr >= q1 - fence) & (arr <= q3 + fence)]

scenarios = [
    ("Elderly added", base_low, base_high, base_low, high_with_elderly),
    ("Sensor errors added", base_low, base_high, low_with_sensor, base_high),
    ("Both added", base_low, base_high, low_with_sensor, high_with_elderly),
]

fig, axes = plt.subplots(3, 3, figsize=(13, 10))
np.random.seed(SEED)

for row_i, (scenario, orig_lo, orig_hi, cont_lo, cont_hi) in enumerate(scenarios):
    lo_iqr = iqr_remove(cont_lo)
    hi_iqr = iqr_remove(cont_hi)

    _, p_orig = stats.ttest_ind(orig_lo, orig_hi)
    _, p_cont = stats.ttest_ind(cont_lo, cont_hi)
    _, p_iqr  = stats.ttest_ind(lo_iqr,  hi_iqr)

    for col_j, (title, l, h, p_val) in enumerate([
        ("Original",                       orig_lo, orig_hi, p_orig),
        (f"With outliers\n({scenario})",   cont_lo, cont_hi, p_cont),
        ("After IQR removal",              lo_iqr,  hi_iqr,  p_iqr),
    ]):
        ax = axes[row_i, col_j]
        bp = ax.boxplot([l, h], labels=["Low", "High"], patch_artist=True,
                        medianprops={"color": "black", "lw": 2})
        bp["boxes"][0].set_facecolor("steelblue")
        bp["boxes"][1].set_facecolor("tomato")
        for k, arr in enumerate([l, h]):
            jitter = np.random.uniform(-0.07, 0.07, len(arr))
            ax.scatter(np.full(len(arr), k + 1) + jitter,
                       arr, alpha=0.5, s=18, zorder=3,
                       color=["steelblue", "tomato"][k])
        stars = ("***" if p_val < 0.001 else
                 "**"  if p_val < 0.01  else
                 "*"   if p_val < 0.05  else "ns")
        ax.set_title(f"{title}\np={p_val:.3f}  {stars}", fontsize=9)
        ax.set_ylabel("Mean HR (BPM)")

plt.suptitle("Impact of different outlier types — low vs high stress comparison",
             fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

### Detecting outliers

An outlier is a value far from the rest of the data. "Far" needs a definition. We will discuss 3 different ones in this section:
1. IQR: already defined and used above. This method uses use **quartiles**. Sort the data and split it into four equal parts: Q1 is the value below which 25% of the data falls, Q2 is the median (50%), Q3 is the 75% point. The **interquartile range (IQR)** is the the width of the middle half of the data: `IQR = Q3 − Q1`. Flag anything below `Q1 − 1.5×IQR` or above `Q3 + 1.5×IQR`. This is the rule behind boxplot whiskers, so any dot plotted separately is an IQR outlier.
2. Z-score: How many standard deviations from the mean is a data point: `z = (x − mean) / sd`.
3. Modified z-score: Fixes the z-score's blind spot by swapping in statistics that outliers can't distort. Instead of the mean, use the median. Instead of the standard deviation, use the MAD — the median distance from the median, `MAD = median(|x − median|)`.

In [ ]:
# ── 2.3  Outlier detection methods compared ─────────────────────────────
def report_outliers(arr, label):
    q1, q3   = np.percentile(arr, [25, 75])
    iqr_mask = (arr < q1 - 1.5*(q3-q1)) | (arr > q3 + 1.5*(q3-q1))

    # CODE HERE START
    # 1. Z-score: distance from the mean in SDs. Flag |z| > 3.
    # 2. Modified z-score: 0.6745 * (x - median) / MAD, where
    #    MAD = median(|x - median|). Flag |mz| > 3.5.

    # z_mask =
    # mz_mask =

    # CODE HERE END

    print(f"\n{label}  (n={len(arr)})")
    for method, mask in [("IQR",              iqr_mask),
                         ("Z-score |z|>3",    z_mask),
                         ("Modified z-score", mz_mask)]:
        vals = ", ".join(f"{v:.1f}" for v in arr[mask]) if mask.any() else "none"
        print(f"  {method:<22}  {mask.sum()} flagged  [{vals}]")

report_outliers(high_with_elderly, "High-stress + elderly participants")
report_outliers(low_with_sensor,   "Low-stress + sensor errors")


**Q2 —**

a. Change `N_ELDERLY` and `N_SENSOR`. At what value does the addition of outliers participants flip the t-test from significant to not significant? Does your answer change with `N_SAMPLES`?  
b. How many of the three elderly participants does each method flag? Should we remove them? What does removing them say about which population our results apply to?  
c. The sensor values (200–220 BPM) are physiologically impossible during an exam. Is removing those a different decision from removing the elderly participants? Explain the distinction.  
d. Looking at the two detection methods in cell 2.3, which is most appropriate when the outliers are sensor errors?

> *Your answer here*

---
# PART II. Real Data — Does Wealth Predict How Long You Live?

We now move to real data. The question we want to answer is simple:

**Does a country's wealth predict how long its citizens live?**

To answer this we will use the Gapminder dataset — life expectancy, GDP per capita, and population for 142 countries in 2007.

---
## Section 3: What does the data look like? <a id='distributions'></a>

Let's just run the correlation between wealth (measured by GDP) and lifespan.

In [ ]:
# ── 3.1  Load Gapminder ──────────────────────────────────────────────────────
GAPMINDER_URL = (
    "https://raw.githubusercontent.com/plotly/datasets/"
    "master/gapminderDataFiveYear.csv"
)

gm     = pd.read_csv(GAPMINDER_URL)
gm2007 = gm[gm["year"] == 2007].copy()

r, p = stats.pearsonr(gm2007["gdpPercap"], gm2007["lifeExp"])
print(f"Pearson correlation: r={r:.3f},  p={p:.2e}")


The relationship looks significant. We seem to be done. But can we trust this result?

Before we run any test, we need to ask: is the test even valid here? Every statistical test has assumptions. If we ignore them we might get a significant result that means nothing, or miss a real effect entirely.

Pearson correlation has assumptions. If they are violated, the result can be
misleading — the p-value too small, the r coefficient being off, or both.

We do not know yet whether our data satisfies these assumptions.
That is why we inspect the data before trusting any test.

**The assumptions:**
1. Both variables are approximately normally distributed
2. The relationship between them is linear
3. No extreme outliers dominating the result
4. Observations are continuous and independent — one real value per country, we can assume this holds

In this section, we check assumptions 1, 2, and 3.

In [ ]:
# ── 3.2  Assumption 1: Normality check through histograms ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col, xlabel in zip(axes,
    ["lifeExp", "gdpPercap"],
    ["Life expectancy (years)", "GDP per capita (USD)"]):
    data  = gm2007[col].dropna()
    ax.hist(data, bins=25, color="steelblue", alpha=0.6, density=True)
    kde_x = np.linspace(data.min(), data.max(), 300)
    ax.plot(kde_x, stats.gaussian_kde(data)(kde_x), color="tomato", lw=2)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Density")
    ax.set_title(f"{col}  (skew={data.skew():.2f})")

plt.suptitle("Are the variables normally distributed?", fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
# ── 3.3  Assumption 2: Linearity check through scatter plot ───────────────────────────────────────

# CODE START HERE
# 1. color the outliers in the plot

fig, ax = plt.subplots(figsize=(7, 5))

ax.scatter(gm2007["gdpPercap"], gm2007["lifeExp"],
           alpha=0.5, s=25, color="steelblue")
ax.set_xlabel("GDP per capita (USD)")
ax.set_ylabel("Life expectancy (years)")
ax.set_title("Assumption 2: is the relationship linear?")
plt.tight_layout()
plt.show()

# CODE END HERE


In [ ]:
# ── 3.4  Assumption 1: Normality check through QQ plots and formal test ────────────────────────────────
# If data is normal, points fall on the red line and p >> 0.05 on the normality test.
# Curves away from the line = non-normal in that region.

# CODE START HERE
# 1. plot 2 qq plots, one for life expectancy and one for gdp per capita

# CODE END HERE


In [ ]:
# ── 3.5  Assumption 3: Identify and examine GDP outliers ────────────────────────────────────
q1, q3  = gm2007["gdpPercap"].quantile([0.25, 0.75])
iqr     = q3 - q1
outliers = gm2007[gm2007["gdpPercap"] > q3 + 1.5 * iqr]

print("Countries flagged as GDP outliers (IQR method):")
print(outliers[["country","continent","gdpPercap","lifeExp"]]
      .sort_values("gdpPercap", ascending=False)
      .to_string(index=False))
print()


**Q3 —**

a. Do you see linear relationship between GDP and life expectancy? What does the relationship look like?  
b. Is life expectancy approximately normally distributed across all 142 countries in 2007? What does the QQ plot tell you that the histogram alone does not?  
c. Norway and Singapore are flagged as outliers by the IQR method but are real values. Should you remove them?
d. Which countries have worse life expectancy than expected based on the trend observed in 3.3. and why? (Optional)

> *Your answer here*


---
## Section 4: Transformations — Useful but Dangerous <a id='transformations'></a>

GDP is right-skewed. A log transform can be a fix for right-skewed economic data. It compresses the long right tail and often makes the relationship linear.

Transformations change the scale or shape of your data. There are two fundamentally different types:

**Non-linear transformations** like log change the *shape* of the distribution. They fix skew and non-linearity.

**Linear transformations** like standardisation and min-max normalisation
change the *scale* but not the shape. They are useful when you have multiple predictors on different scales and want coefficients
to be comparable, or when using distance-based models (KNN, PCA, clustering) where larger-scale features otherwise dominate.

**Common transformations**:

| Transformation | Formula | When to use |
|---------------|---------|-------------|
| **Log** | `np.log(x)` | Right-skewed data; multiplicative relationships |
| **Standardisation** | `(x − mean) / std` | Different units; distance-based models |
| **Min-max normalisation** | `(x − min) / (max − min)` | When you need values in [0, 1] |
| **Within-group scaling** | x/group mean | Compare relative values or trajectories within groups |

However, transformations also change what numbers *mean*. That can mislead you just as badly as not transforming at all.

This section shows:
- How to change shape of your data using a log transformation  
- How to change scale of your data using linear transformations  
- The dangers that come with transformations in general

Before we start, let's see how transformations behave on normally distributed data.


In [ ]:
# CODE START HERE
# 1. generate normally distributed data
# 2. plot the distribution
# 3. apply 2 transformations of your choosing
# 4. plot the transformed data

# CODE END HERE

Now back to our data

In [ ]:
# ── 4.1  Log transform and re-check assumptions ───────────────────────────────
gm2007 = gm2007.copy()
gm2007["log_gdp"] = np.log(gm2007["gdpPercap"])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Histogram
data = gm2007["log_gdp"]
axes[0].hist(data, bins=25, color="steelblue", alpha=0.6, density=True)
kde_x = np.linspace(data.min(), data.max(), 300)
axes[0].plot(kde_x, stats.gaussian_kde(data)(kde_x), color="tomato", lw=2)
axes[0].set_xlabel("log(GDP per capita)")
axes[0].set_title(f"Histogram after transform\nskew={data.skew():.2f}")

# QQ plot
(osm, osr), (slope, intercept, _) = stats.probplot(data, dist="norm")
axes[1].scatter(osm, osr, alpha=0.6, s=18, color="steelblue")
axes[1].plot(osm, slope * np.array(osm) + intercept, "r-", lw=1.5)
_, p_norm = stats.normaltest(data)
axes[1].set_title(f"QQ plot after transform\nnormality p={p_norm:.4f}")
axes[1].set_xlabel("Theoretical quantiles")
axes[1].set_ylabel("Sample quantiles")

# Scatter plot
axes[2].scatter(gm2007["log_gdp"], gm2007["lifeExp"],
                alpha=0.5, s=25, color="steelblue")
axes[2].set_xlabel("log(GDP per capita)")
axes[2].set_ylabel("Life expectancy (years)")
axes[2].set_title("Linearity after transform")

plt.suptitle("After log transform — are assumptions better satisfied?", fontsize=12)
plt.tight_layout()
plt.show()

print(f"Skew before: {gm2007['gdpPercap'].skew():.2f}  →  after: {data.skew():.2f}")


In [ ]:
# ── 4.2  Pearson correlation after transformation — log GDP vs life expectancy ────────────────────

# CODE START HERE
# 1. first run the code as is
# 2. remove outliers from the data
# 3. run the code again

# CODE END HERE

r_raw, p_raw = stats.pearsonr(gm2007["gdpPercap"], gm2007["lifeExp"])
r_log, p_log = stats.pearsonr(gm2007["log_gdp"],   gm2007["lifeExp"])

print(f"{'':38}  {'r':>6}  {'p':>10}")
print("─" * 58)
print(f"{'Pearson — raw GDP (assumptions violated)':<38}  {r_raw:>6.3f}  {p_raw:>10.2e}")
print(f"{'Pearson — log GDP (assumptions satisfied)':<38}  {r_log:>6.3f}  {p_log:>10.2e}")

In [ ]:
# ── 4.3  Linear transformations: change of scale ─────────────
log_gdp  = gm2007[["log_gdp"]].values
life_exp = gm2007["lifeExp"].values

# CODE HERE START
# 1. apply standardisation
# 2. apply minmax normalisation

# std_gdp  =
# norm_gdp =

# CODE HERE END

print(f"{'Transformation':<25}  {'skew':>6}  {'r with lifeExp':>15}  {'value range'}")
print("─" * 72)
for label, arr in [
    ("log(GDP)",          log_gdp.flatten()),
    ("Standardised",      std_gdp),
    ("Min-max normalised",norm_gdp),
]:
    r, _ = stats.pearsonr(arr, life_exp)
    print(f"{label:<25}  {pd.Series(arr).skew():>6.2f}  {r:>15.3f}  "
          f"{arr.min():.2f} to {arr.max():.2f}")

So far, we have transformed a variable using the same rule (reference) for every observation. A log transformation changes its shape, while standardisation and min-max normalisation change its overall scale.

There are other kinds of scaling. For example, we can apply within-group scaling that changes the reference point for each group.

In the exercise below, we will look at how GDP changes over time for different countries. Within-group scaling makes the trajectories easier to compare, but comes with an important trade-off.

The example below shows this trade-off using Norway and Mozambique.

In [ ]:
# ── 4.4  Scaling within groups: what it reveals and hides ────────────────
norway     = gm[gm["country"] == "Norway"].sort_values("year")
mozambique = gm[gm["country"] == "Mozambique"].sort_values("year")

# Filter from 1985 onwards
norway     = norway[norway["year"] >= 1985]
mozambique = mozambique[mozambique["year"] >= 1985]

# Within-group normalisation: each country scaled to its own mean
norway["gdp_rel"]     = norway["gdpPercap"]     / norway["gdpPercap"].mean()
mozambique["gdp_rel"] = mozambique["gdpPercap"] / mozambique["gdpPercap"].mean()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: raw GDP — gap is visible, trajectories impossible to compare
ax = axes[0]
ax.plot(norway["year"],     norway["gdpPercap"],     color="steelblue", lw=2, marker="o", label="Norway")
ax.plot(mozambique["year"], mozambique["gdpPercap"], color="tomato",    lw=2, marker="o", label="Mozambique")
ax.set_xlabel("Year")
ax.set_ylabel("GDP per capita (USD)")
ax.set_title("Raw GDP\nGap visible — trajectories hard to compare")
ax.legend()

# Right: normalised — trajectories comparable, gap invisible
ax = axes[1]
ax.plot(norway["year"],     norway["gdp_rel"],     color="steelblue", lw=2, marker="o", label="Norway")
ax.plot(mozambique["year"], mozambique["gdp_rel"], color="tomato",    lw=2, marker="o", label="Mozambique")
ax.set_xlabel("Year")
ax.set_ylabel("GDP relative to country mean")
ax.set_title("Scaled within country\nTrajectories comparable — gap invisible")
ax.legend()

plt.suptitle("PWithin-group scaling: what it reveals and what it hides", fontsize=12)
years = norway["year"].values
axes[0].set_xticks(years)
axes[1].set_xticks(years)
plt.tight_layout()
plt.show()

gap = (norway["gdpPercap"].values[-1] / mozambique["gdpPercap"].values[-1])
print(f"In 2007, Norway's GDP per capita was {gap:.1f}× Mozambique's.")

**Q4 —**

a. Describe the skew and distribution of raw GDP data. How is it after the log transform? Why does the log transform help? What effect and implications does the outlier removal have?

b. Does standardising log(GDP) change its relationship with life expectancy? Why does this make sense given what standardisation actually does?

c. Look at the Norway and Mozambique trajectories in the normalised plot. Do they look similar? Now look at the raw plot. What is the actual difference between them in 2007? What would you conclude about their development if you only saw the normalised version?

d. You want to compare the effect of GDP and population on life expectancy in the same regression model. Should you standardise? What happens to the coefficients if you do not?

> *Your answer here*


---
## Section 5: Misleading Relationships - Can we still be missing something? <a id='misleading'></a>
Does the overall correlation represent every continent? The correlation coefficient, r=0.81, is a single number summarising 142 countries.

But we saw earlier that life expectancy looks bimodal — two clusters.
What if the strong overall correlation is driven by differences *between* continents rather than the relationship *within* each continent?

A researcher who only had European data could see a very different picture from one who only had African data. If that is the case, the overall r=0.81 does not represent anyone equally well.

In [ ]:
# ── 5.1  Misleading: Overall vs within-continent patterns ─────────────────────
continents = sorted(gm2007["continent"].unique())

print(f"{'Continent':<12}  {'r':>6}  {'p':>8}  {'n':>4}")
print("─" * 36)
print(f"{'Overall':<12}  {r_log:>6.3f}  {p_log:>8.4f}  {len(gm2007):>4}")
print("─" * 36)

for continent in continents:
    grp = gm2007[gm2007["continent"] == continent].dropna(
        subset=["log_gdp", "lifeExp"]
    )

    # Pearson correlation is not meaningful with only two observations
    if len(grp) < 3:
        print(f"{continent:<12}  {'NA':>6}  {'NA':>8}  {len(grp):>4}")
        continue

    r_c, p_c = stats.pearsonr(grp["log_gdp"], grp["lifeExp"])
    print(f"{continent:<12}  {r_c:>6.3f}  {p_c:>8.4f}  {len(grp):>4}")

In the Gapminder data, the relationship between wealth and life expectancy exists within every continent — it is just weaker in some than others.

In some datasets the aggregate relationship is far more misleading. An extreme case of this is **Simpson's paradox**: a trend that appears in the full dataset disappears or reverses when you look at subgroups.

We will demonstrate this on a UC Berkeley admissions dataset describing admissions of men and women to different university departments.


In [ ]:
# ── 5.2  Misleading: Simpson's paradox ────────────────────────────────────
url_ucb = ("https://raw.githubusercontent.com/vincentarelbundock/"
           "Rdatasets/master/csv/datasets/UCBAdmissions.csv")
ucb = pd.read_csv(url_ucb)

# Compute admission rates
dept_rates = []
for dept in sorted(ucb["Dept"].unique()):
    d = ucb[ucb["Dept"] == dept]
    tot = d.groupby("Gender")["Freq"].sum()
    adm = d[d["Admit"] == "Admitted"].set_index("Gender")["Freq"]
    rate = (adm / tot * 100).round(1)
    dept_rates.append({
        "dept":        dept,
        "male_rate":   rate.get("Male",   0),
        "female_rate": rate.get("Female", 0),
    })
dept_df = pd.DataFrame(dept_rates)

overall_tot = ucb.groupby("Gender")["Freq"].sum()
overall_adm = ucb[ucb["Admit"]=="Admitted"].groupby("Gender")["Freq"].sum()
overall_rate = (overall_adm / overall_tot * 100).round(1)

# CODE START HERE
# 1. Create a table comparing men and women within departments: display number of applicants, acceptance rate per department per gender as well as overall

# CODE END HERE

# Plot
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Overall — one point per gender
ax = axes[0]
ax.scatter(["Male", "Female"],
           [overall_rate["Male"], overall_rate["Female"]],
           color=["steelblue", "tomato"], s=150, zorder=5)
ax.plot(["Male", "Female"],
        [overall_rate["Male"], overall_rate["Female"]],
        color="black", lw=2)
ax.set_ylabel("Admission rate (%)")
ax.set_ylim(0, 100)
ax.set_title(f"Overall\nMale={overall_rate['Male']}%  Female={overall_rate['Female']}%\n"
             f"Men admitted at higher rate overall")

# Within departments — one line per department
ax = axes[1]
for _, row in dept_df.iterrows():
    ax.plot(["Male", "Female"],
            [row["male_rate"], row["female_rate"]],
            marker="o", lw=2, label=f"Dept {row['dept']}")
ax.set_ylabel("Admission rate (%)")
ax.set_ylim(0, 100)
ax.set_title("Within each department\nThe overall pattern largely disappears")
ax.legend(fontsize=8)

plt.suptitle("Simpson's paradox — UC Berkeley 1973 admissions\n"
             "Overall and within-department comparisons tell different stories",
             fontsize=11)
plt.tight_layout()
plt.show()

**Q5 —**

a. Do the same conclusions apply to all continents? Is there a continent we should not apply any conclusions to?

b. Why is it important to look at data from different perspectives? How can a relationship be absent or even opposite overall to how it looks like within groups?

c. What strategies can you use to prevent drawing wrong conclusions?

> *Your answer here*

---
## Section 6: Hypothesis Testing and Multiple Comparisons <a id='testing'></a>

As we have already seen, statistical tests have *assumptions*. Running the wrong test or ignoring violated assumptions gives you p-values you cannot trust.

The most common problems are:
- Using a test when data are non-normal and samples are small  
- Ignoring unequal variances between groups  
- Running many tests and treating each p-value as if it were independent

This section shows an example of hypothesis testing and multiple testing correction.

Before working with our real data again, let's run a toy example of hypothesis testing.


In [ ]:
# ── 6.1. Keep testing until significant ───────────────────────────────────────
np.random.seed(42)
attempt = 0

In [ ]:
# RUN THIS CELL AGAIN AND AGAIN
attempt += 1

# We draw 60 times from the same distribution but assign 30 samples to one group and 30 to another
x = np.random.normal(100, 10, 30)
y = np.random.normal(100, 10, 30)

# We test whether the two groups have different population means
_, p = stats.ttest_ind(x, y)

print(f"Attempt {attempt}")
print(f"Mean group x: {x.mean():.2f}")
print(f"Mean group y:  {y.mean():.2f}")
print(f"p-value:     {p:.4f}")

if p < 0.05:
    print("★ SIGNIFICANT! ...but we know there is no real difference.")
else:
    print("Not significant. Run the cell again.")

Now let's get back to our real data.

We want to compare life expectancy across continents.

In [ ]:
# ── 6.2  Assumption check for group comparison ────────────────────────────────
print("Shapiro-Wilk normality test per continent:\n")

groups = []

for continent in continents:
    g = (gm2007[gm2007["continent"] == continent]["lifeExp"].dropna())

    groups.append(g)

    # Shapiro-Wilk requires at least 3 observations
    if len(g) < 3:
        print(
            f"  {continent:<12}  "
            f"n={len(g):>3}  "
            f"not enough observations to test normality"
        )
        continue

    _, p_sw = stats.shapiro(g)

    print(f"  {continent:<12}  n={len(g):>3}  p={p_sw:.4f}  "
          f"{'approximately normal' if p_sw > 0.05 else 'not normal'}")


Before, when data was not normally distributed, we transformed it. That is not the only option. We can also use a non parametric test, meaning one that does not assume normality.

In [ ]:
# ── 6.3  Pairwise tests ──────────────────────────────────────────────────────
# we decide to compare 2 continents against each other (for all continent pairs)
pairs  = list(combinations(continents, 2))
raw_ps = []

print("All pairwise Mann-Whitney U tests (life expectancy):\n")
print(f"  {'Pair':<30}  {'p-value':>10}")
print("  " + "─" * 44)
for c1, c2 in pairs:
    g1 = gm2007[gm2007["continent"] == c1]["lifeExp"].dropna()
    g2 = gm2007[gm2007["continent"] == c2]["lifeExp"].dropna()
    _, p = stats.mannwhitneyu(g1, g2, alternative="two-sided")
    raw_ps.append(p)
    print(f"  {c1 + ' vs ' + c2:<30}  {p:>10.4f}")

print()
print(f"  We found {sum(p < 0.05 for p in raw_ps)} significant pairs.")


In [ ]:
# ── 6.4  Multiple comparison correction ──────────────────────────────────────
raw_arr = np.array(raw_ps)

# CODE HERE START
# 1. apply bonferroni correction
# 2. appply Benjamini-Hochberg/FDR procedure

# ... bonf ... =
# ... bh ... =

# CODE HERE END

print(f"  {'Pair':<30}  {'raw p':>7}  {'Bonferroni':>11}  {'BH-FDR':>8}")
print("  " + "─" * 63)
for (c1, c2), rp, bp, bhp in zip(pairs, raw_arr, bonf, bh):
    label = c1 + " vs " + c2
    print(f"  {label:<30}  "
          f"{rp:>7.4f}{' *' if rp  < 0.05 else '  '}  "
          f"{bp:>10.4f}{' *' if bp  < 0.05 else '  '}  "
          f"{bhp:>7.4f}{' *' if bhp < 0.05 else '  '}")

n_raw  = sum(rp  < 0.05 for rp  in raw_arr)
n_bonf = sum(bp  < 0.05 for bp  in bonf)
n_bh   = sum(bhp < 0.05 for bhp in bh)

print()
print(f"  Significant pairs:  raw={n_raw}  →  Bonferroni={n_bonf}  →  BH-FDR={n_bh}")


**Q6 —**

a. For which continents does the Shapiro-Wilk test indicate non-normality? Does this mean you cannot compare them? Does it change which test you can use?   

b. We ran 10 pairwise tests. At α = 0.05, how many false positives would you expect by chance? How does this change for 100 pairs?  

c. One pair that is significant before correction loses significance under Bonferroni but remains significant under BH-FDR. Which correction would you use, and why does the choice matter for this specific research context?

d. Oceania only has 2 samples, does working with this data make sense?

> *Your answer here*


---
# PART III. Optional section: Who are the Olympic Athletes?

We will use the Olympic dataset providing name, sex, age, height, weight, sport, year, and medal of each athlete.

---
## Section 7: Sampling — Who Is In Your Data, and Who Is Not? <a id='sampling'></a>

Every dataset is a sample from some larger population. The critical question is: **which population?**

If your sample is unrepresentative because of how data were collected, which time period they cover, or which subgroup dominates, then your conclusions may not generalise.

When running statistical tests as we have been doing so far, we usually use all the available data. Sampling is then relevant mostly when things about data collection and what population can we generalise to.

When we start training machine learning models for prediction, classification or other objectives (especially in module 3), sampling becomes even more important. We usually do not use all the data for model training but split our dataset and only use a subset for training. In this case, sampling choices of what data gets included into the training have implications beyond generalisability.

We use the **Olympics** dataset to demonstrate three sampling problems:
1. The whole sample is a biased selection (only elite athletes)  
2. Subgroup composition varies by sport  
3. Random row splitting is wrong when the same person appears on multiple rows or with time series data


In [ ]:
# ── 7.1  Load Olympics dataset ───────────────────────────────────────────────
OLYMPICS_URL = (
    "https://raw.githubusercontent.com/rfordatascience/tidytuesday/"
    "master/data/2021/2021-07-27/olympics.csv"
)

oly = pd.read_csv(OLYMPICS_URL)
print("Shape:           ", oly.shape)
print("Year range:      ", oly["year"].min(), "–", oly["year"].max())
print("Unique athletes: ", oly["id"].nunique())
print("Unique sports:   ", oly["sport"].nunique())


In [ ]:
# ── 7.2  Sampling bias: Olympic athletes ≠ general population ────────────────
valid = oly.dropna(subset=["height", "weight"])

print(f"Mean height of Olympic athletes:  {valid['height'].mean():.1f} cm")
print( "World average adult height:       ~170 cm  (WHO estimate)")
print()

sport_height = valid.groupby("sport")["height"].mean().sort_values()
print("5 shortest-athlete sports:")
print(sport_height.head(5).round(1).to_string())
print()
print("5 tallest-athlete sports:")
print(sport_height.tail(5).round(1).to_string())


In [ ]:
# ── 7.3  Random sampling ─────────────────────────

SAMPLE_FRAC = 0.5    # fraction of athletes to keep

summer = valid[valid["season"] == "Summer"].drop_duplicates(subset="id")

random_sample     = summer.sample(frac=SAMPLE_FRAC, random_state=SEED)

print(f"Full dataset:       {len(summer):>6} athletes, {summer['sport'].nunique()} sports")
print(f"Random sample:      {len(random_sample):>6} athletes, {random_sample['sport'].nunique()} sports")
print()

# Compare sport representation for top 5 sports
full_pct  = summer["sport"].value_counts(normalize=True)
rand_pct  = random_sample["sport"].value_counts(normalize=True)

print("Sport representation — top 5 (% of sample):")
print(f"{'Sport':<22}  {'Full':>6}  {'Random':>8}")
print("─" * 54)
for sp in full_pct.head(5).index:
    f = full_pct.get(sp, 0) * 100
    r = rand_pct.get(sp, 0) * 100
    print(f"{sp:<22}  {f:>5.1f}%  {r:>7.1f}%")


In [ ]:
# ── 7.4  The per-person split problem ────────────────────────────────────────
# The same athlete can appear on multiple rows:
# - multiple events within the same Games
# - participation in multiple Games
#
# If we randomly split rows, the same athlete can therefore appear
# in both the training and test sets.

from sklearn.model_selection import train_test_split as sk_split

# ── How often do athletes repeat? ────────────────────────────────────────────
# Number of rows per athlete
rows_per_athlete = oly.groupby("id").size()
athletes_multiple_rows = rows_per_athlete[rows_per_athlete > 1]

# Number of distinct Games per athlete
games_per_athlete = (oly[["id", "year", "season"]].drop_duplicates().groupby("id").size())

athletes_multiple_games = games_per_athlete[games_per_athlete > 1]

print(f"Total athletes:                   {oly['id'].nunique():>7}")
print(
    f"Athletes with multiple rows:      {len(athletes_multiple_rows):>7}  "
    f"({len(athletes_multiple_rows) / oly['id'].nunique() * 100:.1f}%)"
)
print(
    f"Athletes appearing in 2+ Games:   {len(athletes_multiple_games):>7}  "
    f"({len(athletes_multiple_games) / oly['id'].nunique() * 100:.1f}%)"
)
print()

# ── Prepare feature matrix ───────────────────────────────────────────────────
oly_f = oly.dropna(subset=["height", "weight", "age"]).copy()

oly_f["sex_bin"] = (oly_f["sex"] == "M").astype(int)

X = oly_f[["height", "weight", "age", "sex_bin"]].values
ids = oly_f["id"].values

# ── WRONG: randomly split rows ───────────────────────────────────────────────
row_indices = np.arange(len(X))

train_rows, test_rows = sk_split(row_indices,test_size=0.2,random_state=SEED)

X_tr_rnd = X[train_rows]
X_te_rnd = X[test_rows]

random_train_ids = np.unique(ids[train_rows])
random_test_ids  = np.unique(ids[test_rows])

# Athletes that occur in BOTH sets
leaked_ids = np.intersect1d(random_train_ids, random_test_ids)

# ── RIGHT: split by athlete ID ───────────────────────────────────────────────
unique_ids = np.unique(ids)

train_ids, test_ids = sk_split(unique_ids,test_size=0.2,random_state=SEED)

train_mask = np.isin(ids, train_ids)
test_mask  = np.isin(ids, test_ids)

X_tr_id = X[train_mask]
X_te_id = X[test_mask]

correct_overlap = np.intersect1d(np.unique(ids[train_mask]),np.unique(ids[test_mask]))

# ── Compare ──────────────────────────────────────────────────────────────────
print("WRONG — random row split:")
print(f"  Train: {len(X_tr_rnd):>7} rows, Test: {len(X_te_rnd):>7} rows")
print(f"  Athletes appearing in BOTH sets: {len(leaked_ids):>5}")
print()

print("RIGHT — split by athlete ID:")
print(f"  Train: {len(X_tr_id):>7} rows, Test: {len(X_te_id):>7} rows")
print(f"  Athletes appearing in BOTH sets: {len(correct_overlap):>5}")


In [ ]:
# ── 7.5  Temporal split: train on past, test on future ───────────────────────
SPLIT_YEAR = 2000   # train on < 2000, test on >= 2000

oly_f2 = oly.dropna(subset=["height","weight","age"]).copy()
train_t = oly_f2[oly_f2["year"] <  SPLIT_YEAR]
test_t  = oly_f2[oly_f2["year"] >= SPLIT_YEAR]

print(f"Temporal split at {SPLIT_YEAR}:")
print(f"  Train: {len(train_t):>7} rows  ({train_t['year'].min()}–{train_t['year'].max()})")
print(f"  Test:  {len(test_t):>7} rows  ({test_t['year'].min()}–{test_t['year'].max()})")
print()

# Sport composition shift over time
full_pct  = oly_f2["sport"].value_counts(normalize=True)
train_pct = train_t["sport"].value_counts(normalize=True)
test_pct  = test_t["sport"].value_counts(normalize=True)

print("Sport representation — top sports — training vs test period:")
print(f"{'Sport':<22}  {'Full %':>7}  {'Train %':>8}  {'Test %':>8}")
print("─" * 52)
for sp in full_pct.head(8).index:
    f  = full_pct.get(sp, 0) * 100
    tr = train_pct.get(sp, 0) * 100
    te = test_pct.get(sp, 0) * 100
    print(f"{sp:<22}  {f:>6.1f}%  {tr:>7.1f}%  {te:>7.1f}%")


**Q7 —**

a. If we trained a classification model on this dataset, could we generalise to average population?

b. Which sports are dominant in the random sample? Why does this matter if you then train a classification model? How well can our conclusions generalise?

c. 42.5% of Olympic athletes appear in multiple rows. A colleague randomly splits rows 80/20 and reports 94% test accuracy. Why should you be suspicious of this number? How is temporal leakage similar or different?

d. Some sports were first added to the Olympic program after the year 2000. If you train a model on pre-2000 data, what does this mean for predictions on those sports in the test set?

> *Your answer here*


---
## Wrap-up

This assignment moved through seven core data quality problems.

### Main takeaways

1. **Noise** can hide real effects. More data or better denoising can recover them but over-smoothing can also create effects that were never there.

2. **Outliers** are not all the same. Errors should be removed. Real but rare observations narrow your population when you remove them, and should prompt you to ask: "who are my results actually about?"

3. **Distributions** are rarely normal in the real world. Always plot first. Use QQ plots and formal tests before assuming a parametric test is appropriate.

4. **Transformations** are not free. Log transformations improve linearity but change interpretation. Within-group scaling erases absolute differences. Always report what you did and what it means for your conclusions.

5. **Understanding** is important. A significant result is not the whole story. Always ask whether the effect holds in every subgroup, whether a third variable could explain it, and what your data cannot see.

6. **Multiple testing** inflates false positives. One correction can be strict; another can be a reasonable compromise when some false discoveries are acceptable.

7. **Sampling** determines what you can conclude. European countries are not representative of the world. Psychology students are not representative of an average person's mental health. Randomly splitting rows can be data leakage and produce optimistic metrics that do not generalise.

*End of assignment*
